# 02 — Tourism and Visitor Activity
## Trade Winds & Headwinds: USVI, 2000–2025

Sections:
1. Total arrivals — long-run trend
2. Tourists vs. excursionists
3. Cruise vs. air breakdown
4. Visitor expenditures (nominal)
5. Hotel occupancy and room capacity
6. World Bank cross-check
7. NPS park visitation (VIIS + BUIS)
8. NPS seasonality

**No CPI deflation.** All monetary series are nominal USD.  
**2026 NPS data excluded** (anomalous counts; investigation pending).

In [1]:
import sys
import importlib
from pathlib import Path

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

import os

os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import src.cleaning as _c
import src.charts as _ch

importlib.reload(_c)
importlib.reload(_ch)

from src.cleaning import (
    load_usviber_tour24,
    load_worldbank,
    load_nps_annual,
    load_nps_monthly,
)
from src.charts import (
    line_chart,
    bar_chart,
    stacked_area,
    index_chart,
    monthly_heatmap,
    PALETTE,
)

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
tour = load_usviber_tour24()
wb_arr = load_worldbank("arrivals")
wb_rcpt = load_worldbank("receipts")
viis_ann = load_nps_annual("VIIS")
buis_ann = load_nps_annual("BUIS")
viis_mo = load_nps_monthly("VIIS")
buis_mo = load_nps_monthly("BUIS")

print(f"USVIBER: {tour.year.min()}–{tour.year.max()}  ({len(tour)} obs)")
print(f"VIIS annual: {viis_ann.year.min()}–{viis_ann.year.max()}")
print(f"BUIS annual: {buis_ann.year.min()}–{buis_ann.year.max()}")
print(f"VIIS monthly: {viis_mo.date.min()}–{viis_mo.date.max()}")

USVIBER: 2000–2024  (16 obs)
VIIS annual: 1957–2025
BUIS annual: 1962–2025
VIIS monthly: 1979-01–2025-12


C:\Users\johbr\Data-Analyst\Projects\Trade-Winds-and-Headwinds\src\cleaning.py:237: UserWarning: World Bank arrivals: 1 anomalous value (>5M) set to NaN. Verify source before use.
  warnings.warn(


---
## 1. Total Visitor Arrivals

In [3]:
fig = line_chart(
    tour,
    x="year",
    y=["total_visitors", "tourists", "excursionists"],
    colors=[PALETTE["total"], PALETTE["tourists"], PALETTE["cruise"]],
    names=["Total visitors", "Tourists (overnight)", "Excursionists (day-trip)"],
    title="USVI Visitor Arrivals",
    subtitle="Tourists = overnight stays; excursionists = same-day visitors (cruise + air day-trip)",
    y_label="Visitors (thousands)",
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
    mode="lines+markers",
)
fig.show()

In [4]:
cols = [
    "year",
    "total_visitors",
    "tourists",
    "excursionists",
    "cruise_passengers",
    "air_excursionists",
]
print(tour[cols].set_index("year").to_string())

      total_visitors  tourists  excursionists  cruise_passengers  air_excursionists
year                                                                               
2000         2,516.9     546.7        1,970.2            1,768.4              216.7
2010         2,697.7     576.2        2,121.5            1,858.9              112.9
2011         2,688.4     566.9        2,121.5            2,009.0              112.5
2012         2,646.3     619.2        2,027.2            1,904.5              122.7
2013         2,716.5     590.3        2,126.2            1,998.6              127.6
2014         2,830.4     614.8        2,215.6            2,083.9              131.7
2015         2,653.3     641.9        2,011.4            1,878.8              132.6
2016         2,584.5     667.9        1,916.7            1,776.7              139.9
2017         1,965.1     535.0        1,430.1            1,317.2              112.9
2018         1,923.3     380.9        1,542.5            1,430.7            

---
## 2. Tourists vs. Excursionists — Composition

In [5]:
fig = stacked_area(
    tour,
    x="year",
    y_cols=["tourists", "air_excursionists", "cruise_passengers"],
    names=["Tourists (overnight)", "Air excursionists", "Cruise passengers"],
    colors=[PALETTE["tourists"], PALETTE["air_excursionist"], PALETTE["cruise"]],
    title="Visitor Arrivals by Type",
    subtitle="Stacked area — all values in thousands",
    y_label="Visitors (thousands)",
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.show()

In [6]:
# Tourist share of total arrivals
tour2 = tour.copy()
tour2["tourist_pct"] = tour2["tourists"] / tour2["total_visitors"] * 100
tour2["cruise_pct"] = tour2["cruise_passengers"] / tour2["total_visitors"] * 100

fig = line_chart(
    tour2,
    x="year",
    y=["tourist_pct", "cruise_pct"],
    names=["Tourists % of total", "Cruise % of total"],
    colors=[PALETTE["tourists"], PALETTE["cruise"]],
    title="Visitor Mix: Tourists and Cruise Passengers as Share of Total Arrivals",
    y_label="Percent of total visitors",
    source="USVIBER Annual Tourism Indicator 2024 — author calculations",
    shock_x=[2017, 2020],
)
fig.show()

print(
    tour2[["year", "total_visitors", "tourist_pct", "cruise_pct"]]
    .set_index("year")
    .round(1)
    .to_string()
)

      total_visitors  tourist_pct  cruise_pct
year                                         
2000         2,516.9         21.7        70.3
2010         2,697.7         21.4        68.9
2011         2,688.4         21.1        74.7
2012         2,646.3         23.4        72.0
2013         2,716.5         21.7        73.6
2014         2,830.4         21.7        73.6
2015         2,653.3         24.2        70.8
2016         2,584.5         25.8        68.7
2017         1,965.1         27.2        67.0
2018         1,923.3         19.8        74.4
2019         2,086.9         20.0        68.7
2020           792.8         17.1        55.5
2021         1,053.7         37.6        22.3
2022         1,793.7         22.4        55.9
2023         2,349.9         16.1        66.8
2024         2,507.1         15.2        63.8


> **Note — air excursionists (2020):** The 2020 value (216.7k) is identical to year 2000, which is physically implausible for a COVID year. Cross-check against TOUR20-revised.pdf is pending. The 2020 air excursionist figure is retained but should not be used for per-visitor calculations.

---
## 3. Cruise vs. Air Arrivals

In [7]:
fig = line_chart(
    tour,
    x="year",
    y=["air_visitors", "cruise_passengers"],
    names=["Air arrivals (tourists + air excursionists)", "Cruise passengers"],
    colors=[PALETTE["tourists"], PALETTE["cruise"]],
    title="Air vs. Cruise Arrivals",
    subtitle="Air arrivals = overnight tourists by air + same-day air excursionists",
    y_label="Visitors (thousands)",
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.show()

In [8]:
fig = bar_chart(
    tour,
    x="year",
    y="cruise_ships",
    title="Cruise Ship Calls",
    y_label="Number of cruise ship calls",
    color=PALETTE["cruise"],
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.show()

In [9]:
# Average passengers per cruise ship call
tour2 = tour.copy()
tour2["pax_per_ship"] = tour2["cruise_passengers"] / tour2["cruise_ships"]

fig = bar_chart(
    tour2,
    x="year",
    y="pax_per_ship",
    title="Average Cruise Passengers per Ship Call",
    subtitle="Reflects both ship size trends and itinerary mix",
    y_label="Passengers per call (thousands)",
    color=PALETTE["cruise"],
    source="USVIBER Annual Tourism Indicator 2024 — author calculations",
    shock_x=[2017, 2020],
)
fig.show()
print(
    tour2[["year", "cruise_passengers", "cruise_ships", "pax_per_ship"]]
    .set_index("year")
    .round(1)
    .to_string()
)

      cruise_passengers  cruise_ships  pax_per_ship
year                                               
2000            1,768.4          1014           1.7
2010            1,858.9           680           2.7
2011            2,009.0           698           2.9
2012            1,904.5           667           2.9
2013            1,998.6           626           3.2
2014            2,083.9           655           3.2
2015            1,878.8           609           3.1
2016            1,776.7           569           3.1
2017            1,317.2           427           3.1
2018            1,430.7           481           3.0
2019            1,433.1           529           2.7
2020              440.4           163           2.7
2021              235.5           140           1.7
2022            1,003.1           414           2.4
2023            1,569.3           495           3.2
2024            1,599.8           533           3.0


---
## 4. Visitor Expenditures (Nominal)

In [10]:
fig = line_chart(
    tour,
    x="year",
    y=[
        "expenditures_total",
        "expenditures_tourists",
        "expenditures_cruise_passengers",
        "expenditures_air_excursionists",
    ],
    names=[
        "Total expenditures",
        "Tourist expenditures",
        "Cruise passenger expenditures",
        "Air excursionist (day-trip) expenditures",
    ],
    colors=[
        PALETTE["total"],
        PALETTE["tourists"],
        PALETTE["cruise"],
        PALETTE["air_excursionist"],
    ],
    title="Visitor Expenditures by Category � Nominal",
    subtitle="Millions of current USD. No inflation adjustment.",
    y_label="Expenditures ($M, nominal)",
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
    mode="lines+markers",
)
fig.show()


In [11]:
# Implied expenditure per arrival (nominal)
# Formula: expenditure ($M) / arrivals (thousands) * 1000 = USD per arrival
# These are averages per arrival, not daily spending or per-person totals.
#
# IMPORTANT: spend_per_excursionist uses total excursionists (air + cruise)
# as denominator. For the cruise-specific rate, see spend_per_cruise below.
# The 2020 air_excursionist figure is flagged as suspicious; expenditure
# calculations for 2020 that depend on it should be treated with caution.
tour2 = tour.copy()

tour2["spend_per_tourist"] = (
    tour2["expenditures_tourists"] / tour2["tourists"] * 1000
)  # USD per overnight tourist

tour2["spend_per_cruise"] = (
    tour2["expenditures_cruise_passengers"] / tour2["cruise_passengers"] * 1000
)  # USD per cruise passenger

tour2["spend_per_air_exc"] = (
    tour2["expenditures_air_excursionists"] / tour2["air_excursionists"] * 1000
)  # USD per air excursionist (flag: 2020 value unreliable)

fig = line_chart(
    tour2,
    x="year",
    y=["spend_per_tourist", "spend_per_cruise", "spend_per_air_exc"],
    names=[
        "Per overnight tourist (USD, nominal)",
        "Per cruise passenger (USD, nominal)",
        "Per air excursionist (USD, nominal)",
    ],
    colors=[PALETTE["tourists"], PALETTE["cruise"], PALETTE["air_excursionist"]],
    title="Implied Expenditure per Arrival � Nominal",
    subtitle="Expenditure category / corresponding arrival count. Not daily spending.",
    y_label="USD per arrival (nominal)",
    source="USVIBER Annual Tourism Indicator 2024 � author calculations",
    shock_x=[2017, 2020],
)
fig.show()

# Print table with 2020 flagged
display_cols = ["year", "spend_per_tourist", "spend_per_cruise", "spend_per_air_exc"]
result = tour2[display_cols].set_index("year").round(0)
result.columns = ["Per tourist ($)", "Per cruise pax ($)", "Per air exc ($)"]
print(result.to_string())
print()
print("Note: 2020 air excursionist count is flagged as potentially unreliable.")
print("      Expenditures in nominal/current USD.")


      Per tourist ($)  Per cruise pax ($)  Per air exc ($)
year                                                      
2000          1,307.0               257.0             80.0
2010          1,670.0               309.0            283.0
2011          1,715.0               317.0            292.0
2012          1,752.0               321.0            298.0
2013          1,782.0               326.0            306.0
2014          1,812.0               335.0            312.0
2015          1,810.0               332.0            311.0
2016          1,832.0               341.0            315.0
2017          1,865.0               347.0            318.0
2018          1,896.0               355.0            305.0
2019          2,000.0               376.0            336.0
2020          1,977.0               373.0            350.0
2021          2,067.0               393.0            360.0
2022          2,196.0               408.0            384.0
2023          2,282.0               425.0            392

---
## 5. Hotel Occupancy and Room Capacity

In [12]:
fig = line_chart(
    tour,
    x="year",
    y=["occupancy_rate_pct"],
    names=["Hotel occupancy rate (%)"],
    colors=[PALETTE["occupancy"]],
    title="Hotel Occupancy Rate",
    subtitle="Territory-wide, all hotels and accommodation types",
    y_label="Occupancy rate (%)",
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.update_yaxes(range=[0, 100])
fig.show()

In [13]:
fig = line_chart(
    tour,
    x="year",
    y=["total_rooms", "hotel_rooms", "condo_units"],
    names=["Total rooms/units", "Hotel rooms", "Condo & other units"],
    colors=[PALETTE["total"], PALETTE["rooms"], PALETTE["cruise"]],
    title="Accommodation Inventory — USVIBER Survey",
    subtitle="Annual survey counts. Hotels + condo/vacation rental units.",
    y_label="Rooms or units",
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.show()

# Print the table
print(
    tour[
        [
            "year",
            "total_rooms",
            "hotel_rooms",
            "condo_units",
            "hotel_count",
            "occupancy_rate_pct",
        ]
    ]
    .set_index("year")
    .to_string()
)

      total_rooms  hotel_rooms  condo_units  hotel_count  occupancy_rate_pct
year                                                                        
2000         5008         3947         1061           49                57.7
2010         5008         3878         1131           47                57.1
2011         4889         3759         1130           47                51.6
2012         5030         3874         1157           46                50.9
2013         5016         3897         1120           44                53.8
2014         4974         3856         1118           44                53.8
2015         5060         3903         1157           41                57.1
2016         4842         3819         1023           42                56.7
2017         4886         3820         1065           41                43.9
2018         3255         2854          401           33                31.3
2019         3513         3050          463           34                39.7

---
## 6. World Bank Cross-Check

In [14]:
# Merge USVIBER arrivals with World Bank arrivals for shared years
# WB arrivals are NOT thousands — they are individual counts
wb_arr_clean = wb_arr.dropna(subset=["value"]).copy()
wb_arr_clean["wb_arrivals_k"] = wb_arr_clean["value"] / 1000  # convert to thousands

compare = tour[["year", "total_visitors"]].merge(
    wb_arr_clean[["year", "wb_arrivals_k"]], on="year", how="inner"
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=compare["year"],
        y=compare["total_visitors"],
        mode="lines+markers",
        name="USVIBER total visitors",
        line=dict(color=PALETTE["total"], width=2.5),
    )
)
fig.add_trace(
    go.Scatter(
        x=compare["year"],
        y=compare["wb_arrivals_k"],
        mode="lines+markers",
        name="World Bank arrivals",
        line=dict(color=PALETTE["cruise"], width=2.5, dash="dash"),
    )
)
fig.update_layout(
    title=dict(
        text='USVIBER vs. World Bank: Visitor Arrivals<br><span style="font-size:13px;color:#555">World Bank data ends 2019 (2020 anomaly excluded)</span>',
        font=dict(size=17),
    ),
    yaxis_title="Visitors (thousands)",
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER Annual Tourism Indicator 2024; World Bank WDI ST.INT.ARVL</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
    height=440,
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

compare["ratio"] = (compare["wb_arrivals_k"] / compare["total_visitors"]).round(2)
print("\nWorld Bank / USVIBER ratio (expect ~0.8–1.2 if comparable):")
print(
    compare[["year", "total_visitors", "wb_arrivals_k", "ratio"]]
    .set_index("year")
    .to_string()
)
print("\nNote: Ratio >1 in most years = WB counts more arrivals than USVIBER.")
print(
    "This likely reflects different definitions (WB may include transit and in-territory moves)."
)


World Bank / USVIBER ratio (expect ~0.8–1.2 if comparable):
      total_visitors  wb_arrivals_k  ratio
year                                      
2000         2,516.9        2,396.0    0.9
2010         2,697.7        2,531.0    0.9
2011         2,688.4        2,686.0    1.0
2012         2,646.3        2,640.0    1.0
2013         2,716.5        2,702.0    1.0
2014         2,830.4        2,814.0    1.0
2015         2,653.3        2,643.0    1.0
2016         2,584.5        2,574.0    1.0
2017         1,965.1        1,954.0    1.0
2018         1,923.3        1,922.0    1.0
2019         2,086.9        2,074.0    1.0

Note: Ratio >1 in most years = WB counts more arrivals than USVIBER.
This likely reflects different definitions (WB may include transit and in-territory moves).


---
## 7. NPS Park Visitation

In [15]:
nps = pd.concat([viis_ann, buis_ann]).query("year >= 2000").copy()

fig = go.Figure()
for park, color, name in [
    ("VIIS", PALETTE["nps_viis"], "Virgin Islands NP (St. John)"),
    ("BUIS", PALETTE["nps_buis"], "Buck Island Reef NM (St. Croix)"),
]:
    sub = nps[nps["park"] == park]
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["recreation_visitors"],
            mode="lines+markers",
            name=name,
            line=dict(color=color, width=2.5),
            marker=dict(size=5),
        )
    )

fig.update_layout(
    title=dict(
        text='NPS Recreation Visitation<br><span style="font-size:13px;color:#555">Annual recreation visits — not equivalent to territory-wide tourist arrivals</span>',
        font=dict(size=17),
    ),
    yaxis_title="Recreation visitors",
    yaxis_tickformat=",",
    height=460,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: NPS IRMA Public Use Statistics. BUIS gap 1999–2001. 2026 excluded (anomaly).</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [16]:
# How does NPS recovery compare to USVIBER total arrivals?
# NPS VIIS vs USVIBER total visitors — index to 2016
viis_2016 = viis_ann.query("year==2016")["recreation_visitors"].values[0]
usviber_2016 = tour.query("year==2016")["total_visitors"].values[0]

viis_idx = viis_ann.query("year>=2000").copy()
viis_idx["idx"] = viis_idx["recreation_visitors"] / viis_2016 * 100

tour_idx = tour.copy()
tour_idx["idx_total"] = tour_idx["total_visitors"] / usviber_2016 * 100
tour_idx["idx_tourist"] = (
    tour_idx["tourists"] / tour.query("year==2016")["tourists"].values[0] * 100
)
tour_idx["idx_cruise"] = (
    tour_idx["cruise_passengers"]
    / tour.query("year==2016")["cruise_passengers"].values[0]
    * 100
)

fig = go.Figure()
for label, df, xcol, ycol, color, dash in [
    ("VIIS NP visitors", viis_idx, "year", "idx", PALETTE["nps_viis"], "solid"),
    (
        "Total arrivals (USVIBER)",
        tour_idx,
        "year",
        "idx_total",
        PALETTE["total"],
        "solid",
    ),
    (
        "Overnight tourists",
        tour_idx,
        "year",
        "idx_tourist",
        PALETTE["tourists"],
        "dash",
    ),
    ("Cruise passengers", tour_idx, "year", "idx_cruise", PALETTE["cruise"], "dot"),
]:
    sub = df[[xcol, ycol]].dropna()
    fig.add_trace(
        go.Scatter(
            x=sub[xcol],
            y=sub[ycol],
            mode="lines",
            name=label,
            line=dict(color=color, width=2, dash=dash),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
fig.update_layout(
    title=dict(
        text='Recovery Index: Visitor Series vs. 2016 Baseline<br><span style="font-size:13px;color:#555">Index: 2016 = 100. VIIS NP visits compared with territory-wide arrival series.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Index (2016 = 100)",
    height=480,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Sources: NPS IRMA; USVIBER Annual Tourism Indicator 2024</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [17]:
# Print index table for key years
key_years = [
    2000,
    2010,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
]
viis_tbl = viis_idx[viis_idx["year"].isin(key_years)][
    ["year", "recreation_visitors", "idx"]
].rename(columns={"recreation_visitors": "viis_visits", "idx": "viis_idx"})
tour_tbl = tour_idx[tour_idx["year"].isin(key_years)][
    [
        "year",
        "total_visitors",
        "tourists",
        "cruise_passengers",
        "idx_total",
        "idx_tourist",
        "idx_cruise",
    ]
]
tbl = viis_tbl.merge(tour_tbl, on="year", how="outer").set_index("year")
print(tbl.round(1).to_string())

      viis_visits  viis_idx  total_visitors  tourists  cruise_passengers  idx_total  idx_tourist  idx_cruise
year                                                                                                        
2000       703992     171.1         2,516.9     546.7            1,768.4       97.4         81.9        99.5
2010       493477     120.0         2,697.7     576.2            1,858.9      104.4         86.3       104.6
2015       438372     106.6         2,653.3     641.9            1,878.8      102.7         96.1       105.7
2016       411343     100.0         2,584.5     667.9            1,776.7      100.0        100.0       100.0
2017       304408      74.0         1,965.1     535.0            1,317.2       76.0         80.1        74.1
2018       112287      27.3         1,923.3     380.9            1,430.7       74.4         57.0        80.5
2019       133398      32.4         2,086.9     418.0            1,433.1       80.7         62.6        80.7
2020       167540  

---
## 8. NPS Seasonality

In [18]:
# VIIS monthly heatmap — 2000–2025
viis_mo_plot = viis_mo.query("year >= 2000").copy()
viis_mo_plot["year"] = viis_mo_plot["year"].astype(int)
viis_mo_plot["month"] = viis_mo_plot["month"].astype(int)
viis_mo_plot["visits"] = viis_mo_plot["recreation_visitors"].astype(float)

fig = monthly_heatmap(
    viis_mo_plot.dropna(subset=["visits"]),
    year_col="year",
    month_col="month",
    value_col="visits",
    title="Virgin Islands NP — Monthly Recreation Visitors",
    subtitle="Newest year at top. Blank cells = missing data.",
    source="NPS IRMA Public Use Statistics",
    colorscale="Blues",
    height=700,
)
fig.show()

In [19]:
buis_mo_plot = buis_mo.query("year >= 2000").copy()
buis_mo_plot["year"] = buis_mo_plot["year"].astype(int)
buis_mo_plot["month"] = buis_mo_plot["month"].astype(int)
buis_mo_plot["visits"] = buis_mo_plot["recreation_visitors"].astype(float)

fig = monthly_heatmap(
    buis_mo_plot.dropna(subset=["visits"]),
    year_col="year",
    month_col="month",
    value_col="visits",
    title="Buck Island Reef NM — Monthly Recreation Visitors",
    subtitle="Newest year at top. Blank cells = missing. Gap 1999–2001.",
    source="NPS IRMA Public Use Statistics",
    colorscale="Oranges",
    height=700,
)
fig.show()

In [20]:
# Average seasonal profile by era
# Bins use right-closed intervals; 2017 gets its own bin as the hurricane shock year.
viis_mo2 = viis_mo.query("year >= 2000").copy()
viis_mo2["era"] = pd.cut(
    viis_mo2["year"],
    bins=[1999, 2016, 2017, 2019, 2021, 2025],
    labels=[
        "Pre-hurricane (2000�2016)",
        "Hurricane shock (2017)",
        "Initial recovery (2018�2019)",
        "COVID shock (2020�2021)",
        "Recent (2022�2025)",
    ],
)
seasonal = (
    viis_mo2.dropna(subset=["recreation_visitors", "era"])
    .groupby(["era", "month"])["recreation_visitors"]
    .mean()
    .reset_index()
)
month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]
era_colors = [
    PALETTE["total"],
    "#b22222",
    PALETTE["tourists"],
    PALETTE["cruise"],
    PALETTE["rooms"],
]

fig = go.Figure()
for era, color in zip(seasonal["era"].cat.categories, era_colors):
    sub = seasonal[seasonal["era"] == era].sort_values("month")
    if sub.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=[month_labels[m - 1] for m in sub["month"]],
            y=sub["recreation_visitors"],
            mode="lines+markers",
            name=str(era),
            line=dict(color=color, width=2),
            marker=dict(size=5),
        )
    )

fig.update_layout(
    title=dict(
        text="VIIS � Average Monthly Estimated Recreation Visits by Era<br><span style='font-size:13px;color:#555'>Mean monthly visits per era. 2017 shown separately: annual data mixes pre- and post-hurricane months.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Average monthly estimated recreation visits",
    yaxis_tickformat=",",
    height=460,
    annotations=[
        dict(
            xref="paper", yref="paper", x=0, y=-0.08,
            text="<span style='color:#888;font-size:11px'>Source: NPS IRMA Public Use Statistics. 2003�2021 and 2022�2025 use different estimation methods.</span>",
            showarrow=False, xanchor="left",
        )
    ],
)
fig.show()
